# 01 — Exploratory Data Analysis: CICIDS2017

This notebook characterises the CICIDS2017 dataset before modelling.

**Goals:**
1. Understand class distribution and imbalance severity
2. Identify missing/infinite values
3. Analyse feature distributions and correlations
4. Motivate SMOTE and Focal Loss design choices

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
print('Libraries loaded')

In [ ]:
# Load dataset (use sample if full CICIDS not yet downloaded)
raw_dir = '../data/raw'
sample_file = '../data/sample/sample_traffic.csv'

csv_files = glob.glob(os.path.join(raw_dir, '*.csv'))

if csv_files:
    dfs = [pd.read_csv(f, low_memory=False) for f in csv_files]
    df = pd.concat(dfs, ignore_index=True)
    df.columns = df.columns.str.strip()
    print(f'Full dataset loaded: {df.shape}')
else:
    df = pd.read_csv(sample_file)
    print(f'[WARNING] Using sample data ({df.shape}) — download CICIDS2017 for real EDA')

print(f'Columns: {len(df.columns)}')
df.head(3)

In [ ]:
# ── Class Distribution ─────────────────────────────────────────────────
label_col = 'Label' if 'Label' in df.columns else df.columns[-1]
class_counts = df[label_col].str.strip().value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
class_counts.plot(kind='bar', ax=axes[0], color='#185FA5', alpha=0.85)
axes[0].set_title('Class Distribution (raw count)')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Log scale
class_counts.plot(kind='bar', ax=axes[1], color='#E24B4A', alpha=0.85, logy=True)
axes[1].set_title('Class Distribution (log scale)')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count (log)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../results/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nImbalance ratio (majority/minority):',
      f'{class_counts.max() / class_counts.min():.0f}x')

In [ ]:
# ── Missing & Infinite Values ──────────────────────────────────────────
numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_counts = np.isinf(df[numeric_cols]).sum()
nan_counts = df[numeric_cols].isna().sum()

problem_cols = inf_counts[inf_counts > 0].to_dict()
problem_cols.update(nan_counts[nan_counts > 0].to_dict())

print(f'Columns with inf values: {inf_counts[inf_counts > 0].to_dict()}')
print(f'Columns with NaN values: {nan_counts[nan_counts > 0].to_dict()}')
print(f'\nTotal inf values: {np.isinf(df[numeric_cols]).sum().sum():,}')
print(f'Total NaN values: {df[numeric_cols].isna().sum().sum():,}')

In [ ]:
# ── Feature Correlation Heatmap (top 20 most variable features) ────────
from src.data.feature_engineering import CICIDS_FEATURES, clean_dataframe

df_clean = clean_dataframe(df.copy())
available_feats = [f for f in CICIDS_FEATURES if f in df_clean.columns]

# Select top-20 features by variance (most informative)
variances = df_clean[available_feats].var().sort_values(ascending=False)
top_20_feats = variances.head(20).index.tolist()

corr_matrix = df_clean[top_20_feats].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=False,
            linewidths=0.2, ax=ax)
ax.set_title('Feature Correlation Heatmap (Top 20 by Variance)')
plt.tight_layout()
plt.savefig('../results/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature Distributions by Class ────────────────────────────────────
key_features = ['Flow Duration', 'Flow Packets/s', 'Flow Bytes/s',
                'Fwd Packet Length Mean', 'SYN Flag Count']
key_features = [f for f in key_features if f in df_clean.columns]

top_classes = class_counts.head(6).index.tolist()
df_top = df_clean[df_clean[label_col].str.strip().isin(top_classes)].copy()
df_top[label_col] = df_top[label_col].str.strip()

fig, axes = plt.subplots(1, len(key_features), figsize=(16, 4))
for ax, feat in zip(axes, key_features):
    for cls in top_classes:
        subset = df_top[df_top[label_col] == cls][feat].dropna()
        subset = subset[np.isfinite(subset)]
        if len(subset) > 0:
            subset.hist(ax=ax, bins=40, alpha=0.5, label=cls, density=True)
    ax.set_title(feat[:20], fontsize=9)
    ax.set_xlabel('')
    ax.legend(fontsize=6)

plt.suptitle('Feature Distributions by Top Attack Classes', fontsize=12)
plt.tight_layout()
plt.savefig('../results/figures/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── EDA Summary ────────────────────────────────────────────────────────
print('=== EDA SUMMARY ===')
print(f'Total samples    : {len(df):,}')
print(f'Total features   : {len(available_feats)}')
print(f'Num classes      : {df[label_col].nunique()}')
print(f'Imbalance ratio  : {class_counts.max()/class_counts.min():.0f}x')
print()
print('Key design implications:')
print('  → Severe imbalance justifies SMOTE + Focal Loss')
print('  → High inf/NaN count justifies clean_dataframe() preprocessing')
print('  → High inter-feature correlation justifies dimensionality in CNN layer')